# Phase 3 — Evaluation Harness

Runs all six Phase 3 eval tasks in order on Colab T4.
See `docs/TASKS.md` T3.1–T3.6 for acceptance criteria.

**Run order (dependency):**
1. T3.1 — build spoof manifest (must finish before T3.2)
2. T3.2 — anti-spoof metrics
3. T3.3 — validate tactic corpus (no GPU; instant)
4. T3.4 — tactic classifier v1 vs v2 macro-F1
5. T3.5 — ASR WER: whisper-tiny vs whisper-small
6. T3.6 — end-to-end latency bench
7. Download all outputs as a zip

**Expected total runtime:** ~45 min on Colab T4.

In [ ]:
# --- VishGuard setup cell (same pattern as notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/ben-blake/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)

if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
    # also put eval/ on path so `from eval.xxx import` works
    repo = str(REPO_DIR)
    if repo not in sys.path: sys.path.insert(0, repo)
    os.chdir(REPO_DIR)

import vishguard; print('vishguard', vishguard.__version__)
import torch; print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## T3.1 — Build anti-spoof eval manifest

Downloads 50 real clips from LibriSpeech dev-clean and generates 50 fake
clips with SpeechT5. Writes `eval/out/spoof/manifest.csv` with columns
`path`, `label`, `source`.

**Expected runtime:** ~5 min on T4.

In [ ]:
!python -m eval.buildSpoofSet \
    --n-real 50 \
    --n-fake 50 \
    --out eval/out/spoof/manifest.csv

In [ ]:
import pandas as pd
manifest = pd.read_csv('eval/out/spoof/manifest.csv')
print(manifest.groupby(['label', 'source']).size().reset_index(name='count').to_string(index=False))
print(f'\nTotal rows: {len(manifest)}')
manifest.head()

## T3.2 — Anti-spoof metrics (accuracy / F1 / EER)

Runs `detectSpoof` on every clip in the manifest and computes accuracy,
precision, recall, F1, and EER per subset (librispeech / speecht5) and overall.
Saves a confusion matrix PNG.

**Expected runtime:** ~10 min on T4 (100 clips × inference).

In [ ]:
!python -m eval.runSpoofEval \
    --manifest eval/out/spoof/manifest.csv \
    --out eval/out/spoof/

In [ ]:
from IPython.display import Image, display
import pandas as pd

metrics = pd.read_csv('eval/out/spoof/metrics.csv')
print(metrics.to_string(index=False))

display(Image('eval/out/spoof/confusion_matrix.png'))

## T3.3 — Validate tactic-classification corpus

The 50-script corpus is already committed to `eval/data/tactics.jsonl`.
This cell validates it and prints label-coverage counts.
No GPU required.

**Expected runtime:** < 5 s.

In [ ]:
!python -m eval.buildTacticSet --corpus eval/data/tactics.jsonl

## T3.4 — Tactic classifier: prompt v1 vs v2 macro-F1

Runs `classifyTactics` with both prompt variants across all 50 scripts.
Qwen/Qwen2.5-3B-Instruct is loaded once (module-level cache).
Outputs per-label F1 and macro-F1 for v1 vs v2 to `eval/out/tactics/metrics.csv`.

**Expected runtime:** ~15 min on T4.

In [ ]:
!python -m eval.runTacticEval \
    --corpus eval/data/tactics.jsonl \
    --out eval/out/tactics/

In [ ]:
import pandas as pd

tactic_metrics = pd.read_csv('eval/out/tactics/metrics.csv')

# Summary: macro-F1 comparison
print('=== Macro-F1 v1 vs v2 ===')
print(tactic_metrics[['variant', 'macro_f1']].to_string(index=False))

# Per-label F1
label_cols = [c for c in tactic_metrics.columns if c.startswith('f1_')]
print('\n=== Per-label F1 ===')
print(tactic_metrics[['variant'] + label_cols].to_string(index=False))

## T3.5 — ASR WER: whisper-tiny vs whisper-small

Streams 10 LibriSpeech dev-clean clips, transcribes each under three
conditions (clean, SNR 20 dB, SNR 10 dB) with both models, computes
normalized WER (punctuation stripped), and saves a grouped bar chart.

**Expected runtime:** ~8 min on T4.

In [ ]:
!python -m eval.runAsrEval --out eval/out/asr/

In [ ]:
from IPython.display import Image, display
import pandas as pd

wer_df = pd.read_csv('eval/out/asr/wer_results.csv')
summary = wer_df.groupby(['model', 'condition'])['wer'].mean().reset_index()
summary.columns = ['model', 'condition', 'mean_wer']
summary['mean_wer'] = summary['mean_wer'].round(4)
print(summary.to_string(index=False))

display(Image('eval/out/asr/wer_bar.png'))

## T3.6 — End-to-end latency benchmark

Generates silent WAV clips of 15 s, 45 s, and 90 s, runs the full
pipeline on each (TTS disabled for speed), and reports per-stage latency.

**Expected runtime:** ~5 min on T4.

In [ ]:
!python -m eval.runLatencyBench --out eval/out/latency/

In [ ]:
import pandas as pd

timings = pd.read_csv('eval/out/latency/timings.csv')
summary = pd.read_csv('eval/out/latency/summary.csv')

print('=== Per-clip timings (seconds) ===')
print(timings.to_string(index=False))

print('\n=== Mean across clips ===')
print(summary.T.rename(columns={0: 'mean_s'}).to_string())

## Download all eval outputs

Zips `eval/out/` and downloads it. Unzip locally into `eval/out/`, then commit
the CSVs and PNGs and update `docs/TASKS.md` with the actual numbers.

In [ ]:
import shutil
from pathlib import Path

zip_path = shutil.make_archive('/content/phase3_eval_out', 'zip', 'eval/out')
print(f'Archive: {zip_path}  ({Path(zip_path).stat().st_size / 1024:.0f} KB)')

from google.colab import files
files.download(zip_path)

## Acceptance checklist

Fill in after running all cells and downloading the zip.

- [ ] T3.1 — `eval/out/spoof/manifest.csv` exists with 100 rows (50 real + 50 fake)
- [ ] T3.2 — `eval/out/spoof/metrics.csv` + `confusion_matrix.png` written
- [ ] T3.3 — `buildTacticSet` prints `Validation: PASS`, all 10 labels covered
- [ ] T3.4 — `eval/out/tactics/metrics.csv` written; note v1 vs v2 macro-F1 gap
- [ ] T3.5 — `eval/out/asr/wer_results.csv` + `wer_bar.png` written
- [ ] T3.6 — `eval/out/latency/timings.csv` + `summary.csv` written

After downloading, run locally:
```bash
unzip phase3_eval_out.zip -d eval/out/
git add eval/out/
git commit -m "feat(P3): add eval outputs from Colab T4 run"
```
Then update `docs/TASKS.md` T3.1–T3.6 status to DONE with the actual numbers.